# Install Dependencies

Packages Used

- Langchain
- Chroma
- PyPdf

In [ ]:
%pip install -q langchain-community langchain-google-genai langchain-chroma pypdf

# Importing all the Needed Modules

In [ ]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import CharacterTextSplitter
from langchain_google_genai import GoogleGenerativeAIEmbeddings, ChatGoogleGenerativeAI
from langchain_chroma import Chroma
import os
from getpass import getpass

# Paste your AI API Key

In [ ]:
os.environ["GOOGLE_API_KEY"] = getpass("Enter your Google API key: ")

# Loading the PDF and Chunking

In [ ]:
pages = PyPDFLoader("document.pdf").load()
chunks = CharacterTextSplitter(chunk_size=700, chunk_overlap=100).split_documents(pages)
print(f"{len(pages)} pages, {len(chunks)} chunks")

# Embed the Chunks and Store in the Vector DB

In [ ]:
embeddings = GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-001")
db = Chroma.from_documents(chunks, embeddings, collection_name="pdf_rag")
retriever = db.as_retriever(search_kwargs={"k": 3})

# Retrieval and Generation

In [ ]:
llm = ChatGoogleGenerativeAI(model="gemini-3.5-flash-lite")

def ask(question):
    docs = retriever.invoke(question)
    context = "\n\n".join(
        f"[Page {d.metadata.get('page', 0) + 1}] {d.page_content}" for d in docs
    )
    system_prompt="Answer using only the context. If the answer is absent, say you don't know."
    prompt = f"{system_prompt}\n\nContext:\n{context}\n\nQuestion: {question}"
    return llm.invoke(prompt).content

In [ ]:
print(ask("What is this PDF about?"))